# UTS Data Acquisition & Management — UTS_DAM_2501010174
**Topik:** Pariwisata Bali per kabupaten/kota  
**Alur:** (a) berkas dataset → (b) API Open-Meteo → (c) scraping situs pemerintah (BPS) → gabungan (merge) → visual  
Struktur: `data/raw` (data mentah, tidak diubah) · `Process` (hasil olah) · `docs` (provenance, checklist, bukti robots.txt)

## 0. Konfigurasi & fungsi bantu
**Sebelum menjalankan:** notebook akan meminta **email pribadi** sesuai arahan dosen untuk dicantumkan pada User-Agent. Siapkan satu dataset pariwisata Bali dari portal data resmi dalam format CSV/Excel dengan minimal **100 baris** dan memuat sekurang-kurangnya 5 kabupaten/kota Bali. Unduh dataset terlebih dahulu, lalu unggah saat diminta notebook.

In [ ]:
import os, re, io, sys, json, time, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

try:
    from dotenv import load_dotenv
    load_dotenv()                      # membaca .env (tidak ikut di-upload)
except ImportError:
    pass

# ---------- Identitas ----------
NAMA = "Francesco Nera Alfi Nanggor"
KELAS, PRODI = "F", "Informatika (Management Data Analyst)"
NIM  = "2501010174"
IN_COLAB = "google.colab" in sys.modules
# Email pribadi diminta langsung jika belum disediakan melalui environment.
EMAIL = os.getenv("EMAIL_PRIBADI", "").strip()
if not EMAIL:
    EMAIL = input("Masukkan email pribadi sesuai arahan dosen: ").strip()
if "@" not in EMAIL:
    raise ValueError("Masukkan alamat email pribadi yang valid.")
UA = f"INSTIKI-BIFP243-UTS/1.0 ({EMAIL}; {NAMA}; NIM {NIM})"
TIMEOUT, JEDA = 15, 2.0                # detik

# ---------- Folder ----------
RAW, PROC, DOCS = Path("data/raw"), Path("Process"), Path("docs")
for p in (RAW, PROC, DOCS): p.mkdir(parents=True, exist_ok=True)

# ---------- Dataset berkas ----------
BALI = ["jembrana", "tabanan", "badung", "denpasar", "gianyar", "klungkung", "bangli", "karangasem", "buleleng"]
DATASET_FILE = None
DATASET_URL = DATASET_SUMBER = DATASET_LISENSI = None
READ_KW = dict(sep=None, engine="python", encoding="utf-8", skiprows=0, na_values=["-", "NA", "n/a", ""])
SHEET = 0
KOL_WILAYAH = "ISI_OTOMATIS"
KOL_NILAI   = "ISI_OTOMATIS"

# ---------- Sumber scraping: situs pemerintah (BPS) - dicoba berurutan sampai ada yang lolos robots.txt & berisi tabel ----------
SCRAPE_KANDIDAT = [
    "https://bali.bps.go.id/en/statistics-table/3/TjBWSldsWklZVnBXU1RKcVlsVTNSRXhpU2tzclFUMDkjMw==/jumlah-akomodasi--kamar--dan-tempat-tidur-yang-tersedia-pada-hotel-bintang-menurut-kabupaten-kota-di-provinsi-bali--2023.html?year=2023",
    "https://bali.bps.go.id/en/statistics-table/1/MTg4IzE=/penduduk-laju-pertumbuhan-distribusi-persentase-penduduk-kepadatan-penduduk-dan-rasio-jenis-kelamin-menurut-kabupaten-kota-di-provinsi-bali-2022.html",
    "https://bali.bps.go.id/en/statistics-table/1/NzIjMQ==/luas-lahan-per-kabupaten-kota-menurut-penggunaannya-di-provinsi-bali-2017.html",
    "https://sensus.bps.go.id/topik/tabular/sp2022/189/18/2",
]
SCRAPE_SUMBER = "BPS (situs pemerintah)"
SCRAPE_LISENSI = "Data publik BPS; dipakai non-komersial untuk tugas kuliah dengan mencantumkan sumber (cek ketentuan hak cipta di situs BPS)"

# ---------- Provenance ----------
PROV_FILE = DOCS / "provenance.csv"
def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(8192), b""): h.update(blok)
    return h.hexdigest()

def catat_provenance(nama, url, path, lisensi, metode):
    baris = {"sumber": nama, "url": url, "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
             "lisensi": lisensi, "metode": metode, "berkas": str(path),
             "ukuran_byte": Path(path).stat().st_size, "md5": md5_file(path)}
    df = pd.DataFrame([baris])
    if PROV_FILE.exists():
        lama = pd.read_csv(PROV_FILE)
        df = pd.concat([lama[lama["sumber"] != nama], df], ignore_index=True)  # satu baris per sumber
    df.to_csv(PROV_FILE, index=False)
    return baris

# ---------- Request sopan: timeout + UA + jeda + retry ----------
def get_sopan(url, params=None, maks_coba=4):
    for i in range(1, maks_coba + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA); return r
            if r.status_code in (429, 500, 502, 503, 504):   # layak di-retry
                tunggu = 2 ** i
                print(f"  status {r.status_code}, retry {i}/{maks_coba} dalam {tunggu}s"); time.sleep(tunggu); continue
            print(f"  status {r.status_code} -> berhenti (tidak di-retry)"); r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError) as e:
            tunggu = 2 ** i; print(f"  {type(e).__name__}, retry {i}/{maks_coba} dalam {tunggu}s"); time.sleep(tunggu)
    raise RuntimeError(f"Gagal mengambil {url}")
print("UA:", UA)

### Akuisisi (a): dataset pariwisata Bali minimal 100 baris
Unduh berkas Excel dari halaman resmi Dinas Pariwisata Provinsi Bali (tautan disediakan pada sel berikutnya). Unggah berkas CSV/Excel ke notebook. Notebook memeriksa bahwa satu sheet/berkas memiliki **minimal 100 baris** dan sekurang-kurangnya **5 kabupaten/kota Bali**. Berkas mentah disimpan apa adanya di `data/raw`. Jika syarat tidak terpenuhi, notebook menghentikan proses dan memberi pesan yang jelas; data tidak dibuat-buat atau ditambah secara artifisial.

In [ ]:
# Sumber dataset yang direkomendasikan: Dinas Pariwisata Provinsi Bali.
# Halaman menyediakan berkas Excel Statistik Pariwisata Bali 2025.
URL_REKOMENDASI = "https://disparda.baliprov.go.id/buku-statistik-pariwisata-bali-tahun-2025"
print("Unduh dataset dari halaman resmi ini:", URL_REKOMENDASI)
print("Pilih berkas Excel yang memiliki minimal 100 baris data dan kolom kabupaten/kota.")

if IN_COLAB:
    from google.colab import files
    unggah = files.upload()
    if not unggah:
        raise RuntimeError("Belum ada dataset yang diunggah.")
    nama_f = next(iter(unggah))
    DATASET_FILE = RAW / nama_f
    DATASET_FILE.write_bytes(unggah[nama_f])
else:
    kandidat_file = [p for p in RAW.iterdir() if p.suffix.lower() in (".csv", ".xls", ".xlsx")]
    if not kandidat_file:
        raise RuntimeError("Letakkan satu file CSV/Excel di folder data/raw terlebih dahulu.")
    DATASET_FILE = kandidat_file[0]

DATASET_URL = input("Tempel URL halaman dataset yang diunduh: ").strip() or URL_REKOMENDASI
DATASET_SUMBER = "Dinas Pariwisata Provinsi Bali — Data Statistik Pariwisata Bali 2025"
DATASET_LISENSI = input("Tuliskan lisensi/ketentuan penggunaan yang tercantum pada portal (jika tidak ada, tulis 'Tidak dicantumkan'): ").strip() or "Tidak dicantumkan pada halaman yang diperiksa"

# Periksa isi berkas dan pilih sheet yang memenuhi syarat: >=100 baris dan >=5 kabupaten/kota Bali.
def wilayah_bali(x):
    t = str(x).lower()
    return next((b for b in BALI if b in t), None)

def evaluasi_df(df):
    if df is None or len(df) < 100:
        return None
    skor = {c: int(df[c].astype(str).map(wilayah_bali).notna().sum())
            for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])}
    if not skor:
        return None
    c = max(skor, key=skor.get)
    n_wil = int(df[c].astype(str).map(wilayah_bali).nunique())
    if n_wil >= 1:
        return {"kolom_wilayah": c, "n_wilayah": n_wil, "baris": len(df)}
    return None

if DATASET_FILE.suffix.lower() in (".xls", ".xlsx"):
    buku = pd.read_excel(DATASET_FILE, sheet_name=None)
    lolos = [(nm, evaluasi_df(df)) for nm, df in buku.items() if evaluasi_df(df)]
    if not lolos:
        raise ValueError("Dataset Excel belum memenuhi syarat: minimal 100 baris dan minimal 1 kabupaten/kota Bali dalam satu sheet. Pilih sheet/berkas lain.")
    SHEET, ev = lolos[0]
else:
    ev = None
    for enc in ("utf-8", "latin-1"):
        try:
            cek_df = pd.read_csv(DATASET_FILE, sep=None, engine="python", encoding=enc)
            ev = evaluasi_df(cek_df)
            if ev:
                READ_KW["encoding"] = enc
                break
        except Exception:
            pass
    if not ev:
        raise ValueError("Dataset CSV belum memenuhi syarat: minimal 100 baris dan minimal 1 kabupaten/kota Bali.")

KOL_WILAYAH = ev["kolom_wilayah"]
print("DATASET LOLOS:", DATASET_FILE.name)
print("Sheet:", SHEET, "| Baris:", ev["baris"], "| Kabupaten/kota unik:", ev["n_wilayah"])
print("URL:", DATASET_URL, "\nSUMBER:", DATASET_SUMBER, "\nLISENSI:", DATASET_LISENSI)


## Soal 1 — Baca berkas, identifikasi dataset, estimasi storage
Berkas dibaca dengan parameter yang sesuai (`sep`, `encoding`, `skiprows`, `na_values`, `dtype`, `sheet_name`). Berkas mentah **tidak diubah**.

In [ ]:
assert DATASET_FILE.exists(), f"Letakkan dataset hasil unduhan di {DATASET_FILE}"
ext = DATASET_FILE.suffix.lower()
if ext in (".xlsx", ".xls"):
    df_raw = pd.read_excel(DATASET_FILE, sheet_name=SHEET, skiprows=READ_KW["skiprows"], na_values=READ_KW["na_values"])
else:
    df_raw = pd.read_csv(DATASET_FILE, **READ_KW)
df_raw.columns = [str(c).strip() for c in df_raw.columns]   # hanya salinan di memori; berkas asli utuh

# ---- Deteksi otomatis kolom wilayah & kolom nilai bila belum diisi manual ----
if KOL_WILAYAH.startswith("ISI"):
    skor = {c: int(df_raw[c].astype(str).str.lower().map(lambda x: any(b in x for b in BALI)).sum())
            for c in df_raw.columns if not pd.api.types.is_numeric_dtype(df_raw[c])}
    KOL_WILAYAH = max(skor, key=skor.get)
    print("KOL_WILAYAH terdeteksi:", KOL_WILAYAH, f"({skor[KOL_WILAYAH]} baris cocok nama kab/kota Bali)")
if KOL_NILAI.startswith("ISI"):
    kandidat = [c for c in df_raw.select_dtypes("number").columns if not re.search(r"tahun|year|kode|code|^no$|id$", c, re.I)]
    if kandidat:
        KOL_NILAI = kandidat[0]
        print("KOL_NILAI terdeteksi:", KOL_NILAI, "| kandidat lain:", kandidat[1:])
    else:                                              # daftar objek tanpa kolom angka -> indikator = jumlah objek per wilayah
        df_raw["jumlah_objek"] = 1; KOL_NILAI = "jumlah_objek"
        print("Tidak ada kolom angka -> indikator = jumlah baris/objek per wilayah")

print("Dimensi (baris, kolom):", df_raw.shape)
assert len(df_raw) >= 100, "Dataset minimal 100 baris (syarat soal)"
display(df_raw.head())
display(df_raw.dtypes.to_frame("tipe_data"))
catat_provenance("Dataset berkas", DATASET_URL, DATASET_FILE, DATASET_LISENSI, "Unduh manual (CSV/Excel)")

In [ ]:
# Identifikasi dataset
ukuran_byte = DATASET_FILE.stat().st_size
ukuran_mb = ukuran_byte / 1024 / 1024
identitas = {
    "jenis_data": "Terstruktur (tabular: baris-kolom, skema tetap)",
    "jumlah_baris": df_raw.shape[0], "jumlah_kolom": df_raw.shape[1],
    "kolom_numerik": list(df_raw.select_dtypes("number").columns),
    "kolom_teks": list(df_raw.select_dtypes(exclude="number").columns),
    "ukuran_berkas_MB": round(ukuran_mb, 4),
    "wilayah_unik": int(df_raw[KOL_WILAYAH].nunique()) if KOL_WILAYAH in df_raw else "isi KOL_WILAYAH",
}
# cakupan waktu: cari kolom bernama tahun/bulan/tanggal/periode
kol_waktu = [c for c in df_raw.columns if re.search(r"tahun|year|bulan|month|tanggal|date|periode", c, re.I)]
if kol_waktu:
    k = kol_waktu[0]; identitas["cakupan_waktu"] = f"{df_raw[k].min()} s.d. {df_raw[k].max()} (kolom {k})"
else:
    identitas["cakupan_waktu"] = "ISI MANUAL dari metadata portal"
for k, v in identitas.items(): print(f"{k:18}: {v}")
print("\nSatuan: ISI MANUAL dari deskripsi portal (mis. jumlah kunjungan = orang, luas = km2)")  # <-- ISI

In [ ]:
# Estimasi storage 1 tahun = ukuran berkas x 12 bulan, + cadangan 30%
CADANGAN = 0.30
dasar = ukuran_mb * 12
total = dasar * (1 + CADANGAN)
print(f"Ukuran berkas             : {ukuran_mb:.4f} MB")
print(f"x 12 bulan                : {dasar:.4f} MB")
print(f"+ cadangan {CADANGAN:.0%}            : {total:.4f} MB (= {total/1024:.6f} GB)")
estimasi = pd.DataFrame({"komponen": ["ukuran_berkas_MB", "x12_bulan_MB", "cadangan_30persen_MB", "total_1_tahun_MB"],
                         "nilai": [ukuran_mb, dasar, dasar*CADANGAN, total]})
estimasi.to_csv(DOCS / "estimasi_storage.csv", index=False)

**Asumsi estimasi storage:** (1) dataset diperbarui/diunduh ulang sekali per bulan dan setiap unduhan disimpan sebagai berkas terpisah (versi bulanan) berukuran sama dengan berkas sekarang; (2) tidak ada kompresi; (3) cadangan 30% untuk file hasil olahan (CSV bersih, JSON, HTML), log, dan pertumbuhan data; (4) data API & scraping tidak dihitung terpisah karena ukurannya jauh lebih kecil (dicantumkan pada tabel provenance).

## Soal 2 — Lisensi, PII/quasi-identifier, dan Data Ethics Checklist
Langkah: periksa lisensi & ketentuan portal (dicatat di provenance), klasifikasi kolom, lalu isi checklist 9 butir dengan bukti.

In [ ]:
# Klasifikasi kolom: nama kolom + pola isi (email, telepon, NIK 16 digit)
PII_LANGSUNG = r"nik|nama(?!_?(wilayah|kabupaten|kota|objek|wisata|daerah))|email|telp|telepon|phone|hp|alamat|ktp|npwp"
QUASI = r"tgl|tanggal_lahir|umur|usia|kode_?pos|jenis_kelamin|gender|kecamatan|desa|kelurahan|koordinat|lat|lon"
pola_isi = {"email": r"[\w.+-]+@[\w-]+\.[\w.]+", "telepon": r"(\+62|08)\d{8,12}", "nik": r"\b\d{16}\b"}

hasil = []
for c in df_raw.columns:
    kategori = "Non-PII"
    if re.search(PII_LANGSUNG, c, re.I): kategori = "PII langsung"
    elif re.search(QUASI, c, re.I): kategori = "Quasi-identifier"
    if not pd.api.types.is_numeric_dtype(df_raw[c]):
        contoh = df_raw[c].dropna().astype(str).head(300)
        for jenis, p in pola_isi.items():
            if contoh.str.contains(p, regex=True).any(): kategori = f"PII langsung (pola {jenis})"
    hasil.append({"kolom": c, "kategori": kategori})
klasifikasi = pd.DataFrame(hasil)
display(klasifikasi)
klasifikasi.to_csv(DOCS / "klasifikasi_pii.csv", index=False)
ada_pii = klasifikasi["kategori"].str.startswith("PII").any()
print("Ada PII langsung:", ada_pii)

In [ ]:
# Data Ethics Checklist 9 butir (format Modul 4) — lengkap dengan bukti
prov = pd.read_csv(PROV_FILE)
def bukti(nm, kol): 
    r = prov[prov["sumber"] == nm]; return r.iloc[0][kol] if len(r) else "-"

checklist = pd.DataFrame({
    "Aspek": ["Sumber data dan URL dicantumkan", "Lisensi data diperiksa dan dicatat",
              "robots.txt / Terms of Service diperiksa", "Kolom PII diidentifikasi",
              "PII dihapus, di-mask, atau di-pseudonimisasi", "Prinsip minimalisasi data diterapkan",
              "Potensi bias sampel dianalisis", "Tujuan penggunaan data dinyatakan eksplisit",
              "Rencana retensi dan penghapusan data ada"],
    "Status (Ya/Tidak/NA)": ["Ya", "Ya", "Ya", "Ya",
                             "NA" if not ada_pii else "Ya", "Ya", "Ya", "Ya", "Ya"],
    "Bukti / Catatan": [
        f"docs/provenance.csv — 3 sumber; berkas: {DATASET_URL}; API: archive-api.open-meteo.com; scraping: lihat docs/scraping_meta.json",
        f"Dataset: {DATASET_LISENSI} | Open-Meteo: CC BY 4.0 (atribusi, non-komersial) | Situs scraping: {SCRAPE_LISENSI}. Tangkapan layar lisensi portal: simpan di docs/ (ISI)",
        "docs/robots_check.json (Soal 4) berisi isi robots.txt, hasil can_fetch, dan crawl-delay; ToS Wikimedia/Open-Meteo dibaca (penggunaan wajar, tidak komersial, UA jelas)",
        "docs/klasifikasi_pii.csv — setiap kolom diklasifikasikan (PII langsung / quasi-identifier / non-PII)",
        "Data bersifat agregat per wilayah, tidak ada kolom PII langsung" if not ada_pii else "Kolom PII di-drop/pseudonimisasi pada tahap pembersihan (lihat sel Soal 5)",
        "Hanya kolom yang dibutuhkan untuk merge (wilayah, indikator, cuaca, dan maksimal 3 kolom angka hasil scraping) yang dipakai; API hanya variabel suhu & hujan harian",
        "Bias seleksi: hanya 9 wilayah Bali & 1 tahun; indikator pariwisata dari data resmi mungkin tidak mencakup usaha informal; cuaca diambil di titik ibu kota, bukan rata-rata wilayah",
        "Tujuan: tugas UTS — menganalisis hubungan indikator pariwisata dengan kondisi wilayah & cuaca; tidak dipakai komersial/profiling individu",
        "Data mentah disimpan selama semester berjalan (hingga nilai final), lalu dihapus dari repo & perangkat; hasil gabungan agregat boleh diarsipkan",
    ],
})
checklist.to_csv(DOCS / "data_ethics_checklist.csv", index=False)
display(checklist)

## Soal 3 — Akuisisi via API (Open-Meteo, cuaca historis)
Portal sumber tidak memakai API terbuka yang praktis, maka dipakai **API publik Open-Meteo** (tanpa API key) untuk cuaca harian di ibu kota 9 kabupaten/kota Bali — sesuai topik pariwisata (cuaca memengaruhi kunjungan). Status code & retry ditangani oleh `get_sopan`. Respons JSON mentah disimpan di `data/raw`.

In [ ]:
# Koordinat pendekatan ibu kota tiap kabupaten/kota (hanya untuk titik cuaca)
LOKASI = {
    "Jembrana": (-8.36, 114.62), "Tabanan": (-8.54, 115.12), "Badung": (-8.58, 115.18),
    "Denpasar": (-8.65, 115.22), "Gianyar": (-8.54, 115.33), "Klungkung": (-8.54, 115.40),
    "Bangli": (-8.45, 115.35), "Karangasem": (-8.45, 115.61), "Buleleng": (-8.11, 115.09),
}
API_URL = "https://archive-api.open-meteo.com/v1/archive"
MULAI, AKHIR = "2025-01-01", "2025-12-31"

baris_api = []
for wilayah, (lat, lon) in LOKASI.items():
    params = {"latitude": lat, "longitude": lon, "start_date": MULAI, "end_date": AKHIR,
              "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Asia/Makassar"}
    r = get_sopan(API_URL, params=params)
    js = r.json()
    (RAW / f"cuaca_openmeteo_{wilayah.lower()}.json").write_text(json.dumps(js, ensure_ascii=False), encoding="utf-8")
    d = js["daily"]
    for t, suhu, hujan in zip(d["time"], d["temperature_2m_mean"], d["precipitation_sum"]):
        baris_api.append({"wilayah": wilayah, "tanggal": t, "suhu_rata2_C": suhu, "hujan_mm": hujan})
    print(f"{wilayah}: {len(d['time'])} hari")

cuaca_harian = pd.DataFrame(baris_api)            # normalisasi JSON -> tabel
cuaca_harian["tanggal"] = pd.to_datetime(cuaca_harian["tanggal"])
assert len(cuaca_harian) >= 30
print("Tabel hasil normalisasi:", cuaca_harian.shape)
display(cuaca_harian.head())

# catat provenance (satu berkas gabungan JSON dibuat agar md5 satu sumber)
gabung_json = RAW / "cuaca_openmeteo_semua.json"
gabung_json.write_text(json.dumps(baris_api, default=str, ensure_ascii=False), encoding="utf-8")
catat_provenance("API Open-Meteo", API_URL, gabung_json, "CC BY 4.0 (open-meteo.com/en/license)", "API JSON (tanpa key)")
cuaca_harian.to_csv(PROC / "cuaca_harian_bersih.csv", index=False)
(PROC / "cuaca_harian_bersih.json").write_text(cuaca_harian.to_json(orient="records", date_format="iso", force_ascii=False), encoding="utf-8")

In [ ]:
# Ringkas per wilayah (kunci merge: wilayah)
cuaca_wilayah = (cuaca_harian.groupby("wilayah")
                 .agg(suhu_rata2_C=("suhu_rata2_C", "mean"), hujan_total_mm=("hujan_mm", "sum"),
                      hari_hujan=("hujan_mm", lambda s: int((s >= 1).sum())))
                 .round(2).reset_index())
display(cuaca_wilayah)

## Soal 4 — Scraping etis halaman statis situs pemerintah + bukti robots.txt
Halaman kandidat (situs BPS) dicoba berurutan: (1) cek `robots.txt` → (2) request dengan UA, timeout, jeda → (3) pilih tabel yang memuat nama kab/kota Bali (berbasis isi, bukan class CSS yang rapuh) → (4) simpan HTML mentah, bukti robots, dan metadata. Halaman yang dilarang robots.txt atau dirender JavaScript dilewati.

In [ ]:
def cek_kepatuhan(url, user_agent=UA):
    p = urlparse(url); base = f"{p.scheme}://{p.netloc}"
    rr = requests.get(base + "/robots.txt", headers={"User-Agent": UA}, timeout=TIMEOUT)
    rp = RobotFileParser(); rp.parse(rr.text.splitlines())
    return {"url": url, "waktu_cek": dt.datetime.now().isoformat(timespec="seconds"), "user_agent": user_agent,
            "status_robots_txt": rr.status_code,
            "boleh_diakses": rp.can_fetch(user_agent, url), "boleh_diakses_wildcard": rp.can_fetch("*", url),
            "crawl_delay": rp.crawl_delay(user_agent), "isi_robots_txt_cuplikan": rr.text[:500]}

def hitung_bali(t):
    return int(t.astype(str).apply(lambda col: col.str.lower().map(lambda x: any(b in x for b in BALI))).sum().sum())

bukti_semua, tabel_scrape, TARGET = [], None, None
for url in SCRAPE_KANDIDAT:
    print("Mencoba:", url[:110])
    try: bukti = cek_kepatuhan(url)
    except Exception as e: print("  robots.txt tidak terbaca:", type(e).__name__); continue
    bukti_semua.append(bukti)
    if not bukti["boleh_diakses"]: print("  robots.txt melarang -> lewati"); continue
    try: r = get_sopan(url)
    except Exception as e: print("  gagal mengambil halaman:", type(e).__name__); continue
    try: tabel_semua = pd.read_html(io.StringIO(r.text), thousands=None)   # thousands=None: angka diurai sendiri oleh ke_angka
    except ValueError: tabel_semua = []
    skor_tabel = [hitung_bali(t) for t in tabel_semua]
    if skor_tabel and max(skor_tabel) >= 6:
        TARGET = url
        tabel_scrape = tabel_semua[int(np.argmax(skor_tabel))].copy()
        tabel_scrape.columns = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in tabel_scrape.columns]
        html_path = RAW / "scraping_halaman_pemerintah.html"
        html_path.write_text(r.text, encoding="utf-8")
        catat_provenance("Scraping situs pemerintah", TARGET, html_path, SCRAPE_LISENSI, "Scraping halaman statis (requests+BeautifulSoup)")
        print("  BERHASIL: tabel ditemukan", tabel_scrape.shape); break
    print("  tidak ada tabel kab/kota Bali di HTML awal (kemungkinan dirender JavaScript) -> lewati")

(DOCS / "robots_check.json").write_text(json.dumps(bukti_semua, indent=2, ensure_ascii=False), encoding="utf-8")
bukti_robots = next((b for b in bukti_semua if b["url"] == TARGET), {"boleh_diakses": None})
for b in bukti_semua: print({k: b[k] for k in ("url", "status_robots_txt", "boleh_diakses", "crawl_delay")})
if tabel_scrape is not None:
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "lxml"); print("Judul halaman:", soup.title.get_text(strip=True) if soup.title else "-")
    display(tabel_scrape)

In [ ]:
# Standardisasi: kolom nama wilayah + maksimal 3 kolom angka (format ID "1.234,5" atau EN "1,234.5" dikenali)
def ke_angka(x):
    s = re.sub(r"\[.*?\]", "", str(x)).strip().replace(" ", "")
    if re.fullmatch(r"\d{1,3}(\.\d{3})+(,\d+)?", s): s = s.replace(".", "").replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(,\d{3})+(\.\d+)?", s): s = s.replace(",", "")
    elif re.fullmatch(r"\d+,\d+", s): s = s.replace(",", ".")
    try: return float(s)
    except ValueError: return np.nan

wiki, KET_KOLOM_SCRAPE, SCRAPE_PAKAI = None, {}, None
if tabel_scrape is not None:
    skor_kolom = {c: int(tabel_scrape[c].astype(str).str.lower().map(lambda x: any(b in x for b in BALI)).sum()) for c in tabel_scrape.columns}
    c_nama = max(skor_kolom, key=skor_kolom.get)
    angka = {c: tabel_scrape[c].map(ke_angka) for c in tabel_scrape.columns if c != c_nama}
    angka = {c: v for c, v in angka.items() if v.notna().sum() >= 6 and not re.search(r"year|tahun|no\b", c, re.I)}
    KET_KOLOM_SCRAPE = {f"scrape_{i+1}": c for i, c in enumerate(list(angka)[:3])}   # pemetaan nama -> header asli
    print("Kolom wilayah :", c_nama); print("Pemetaan kolom angka:", KET_KOLOM_SCRAPE)
    if KET_KOLOM_SCRAPE:
        wiki = pd.DataFrame({"nama_scrape": tabel_scrape[c_nama].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip()})
        for k, c in KET_KOLOM_SCRAPE.items(): wiki[k] = angka[c].values
        wiki = wiki[wiki["nama_scrape"].str.lower().map(lambda x: any(b in x for b in BALI))].reset_index(drop=True)  # buang baris total
        wiki = wiki.drop_duplicates("nama_scrape").reset_index(drop=True)
        SCRAPE_PAKAI = SCRAPE_SUMBER
        display(wiki)
        meta_scrape = {"situs": TARGET, "sumber": SCRAPE_SUMBER, "waktu": dt.datetime.now().isoformat(timespec="seconds"), "user_agent": UA,
                       "jeda_detik": JEDA, "jumlah_record": int(len(wiki)), "robots_mengizinkan": bukti_robots["boleh_diakses"],
                       "pemetaan_kolom": KET_KOLOM_SCRAPE, "lisensi": SCRAPE_LISENSI, "diambil_oleh": f"{NAMA} / {NIM}"}
        (DOCS / "scraping_meta.json").write_text(json.dumps(meta_scrape, indent=2, ensure_ascii=False), encoding="utf-8")
        wiki.to_csv(PROC / "scraping_wilayah_bersih.csv", index=False)
        (PROC / "scraping_wilayah_bersih.json").write_text(wiki.to_json(orient="records", force_ascii=False), encoding="utf-8")
        (PROC / "scraping_wilayah_bersih.html").write_text(wiki.to_html(index=False), encoding="utf-8")
if wiki is None:
    print("Scraping tidak menghasilkan tabel -> penggabungan memakai 2 sumber (berkas + API), sesuai syarat 'minimal dua dari tiga'.")

## Soal 5 — Penggabungan (merge) + validasi + simpan
**Kunci merge:** `kunci_wilayah` = nama kabupaten/kota yang sudah dinormalisasi (huruf kecil, tanpa kata "Kabupaten/Kota/Regency/City"). Kunci bermakna (bukan konstanta) dan diverifikasi 1-ke-1 dengan `validate`.

In [ ]:
def kunci(s):                                    # kunci = nama kab/kota Bali baku (mis. "Kota Denpasar" / "Kab. Badung" -> denpasar / badung)
    t = str(s).lower()
    for b in BALI:
        if b in t: return b
    return re.sub(r"[^a-z]", "", re.sub(r"\b(kabupaten|kab\.?|kota|regency|city)\b", "", t))

# Dataset berkas -> agregasi per wilayah (satu baris per wilayah agar merge 1:1)
d = df_raw.copy()
kol_tahun = [c for c in d.columns if re.search(r"tahun|year", c, re.I) and pd.api.types.is_numeric_dtype(d[c])]
if kol_tahun:                                   # bila ada beberapa tahun, pakai tahun terbaru agar merge 1:1 bermakna
    tahun_pakai = d[kol_tahun[0]].max(); d = d[d[kol_tahun[0]] == tahun_pakai]
    print(f"Dipakai tahun terbaru: {tahun_pakai} (kolom {kol_tahun[0]})")
d["kunci_wilayah"] = d[KOL_WILAYAH].map(kunci)
berkas_wil = d.groupby("kunci_wilayah", as_index=False).agg(wilayah_berkas=(KOL_WILAYAH, "first"),
                                                              indikator_pariwisata=(KOL_NILAI, "sum"))
cuaca_wilayah["kunci_wilayah"] = cuaca_wilayah["wilayah"].map(kunci)
if wiki is not None: wiki["kunci_wilayah"] = wiki["nama_scrape"].map(kunci)

# Bukti kunci tidak duplikat
for nama, t in {"berkas": berkas_wil, "api": cuaca_wilayah, **({"scraping": wiki} if wiki is not None else {})}.items():
    print(f"{nama:9}: baris={len(t)}, kunci unik={t['kunci_wilayah'].nunique()}")
    assert t["kunci_wilayah"].is_unique, f"Kunci duplikat di {nama}"

# Merge bertahap dengan validate
m1 = berkas_wil.merge(cuaca_wilayah.drop(columns="wilayah"), on="kunci_wilayah", how="inner", validate="one_to_one")
gabungan = m1.merge(wiki, on="kunci_wilayah", how="inner", validate="one_to_one") if wiki is not None else m1.copy()

print("\nBaris gabungan:", len(gabungan))
print("Duplikat baris :", int(gabungan.duplicated().sum()), "| duplikat kunci:", int(gabungan["kunci_wilayah"].duplicated().sum()))
print("Wilayah tidak cocok (berkas):", sorted(set(berkas_wil.kunci_wilayah) - set(gabungan.kunci_wilayah)))
assert gabungan["kunci_wilayah"].is_unique
if wiki is not None:
    gabungan["indikator_per_scrape_1"] = (gabungan["indikator_pariwisata"] / gabungan["scrape_1"]).round(6)
    print("Arti kolom scrape:", KET_KOLOM_SCRAPE)
display(gabungan)

In [ ]:
# Simpan hasil + metadata
gabungan.to_csv(PROC / "gabungan_pariwisata_bali.csv", index=False)
(PROC / "gabungan_pariwisata_bali.json").write_text(gabungan.to_json(orient="records", force_ascii=False, indent=2), encoding="utf-8")
(PROC / "gabungan_pariwisata_bali.html").write_text(gabungan.to_html(index=False), encoding="utf-8")
df_raw.to_csv(PROC / "dataset_berkas_bersih.csv", index=False)   # salinan bersih; mentah tetap di data/raw

meta_merge = {"dibuat": dt.datetime.now().isoformat(timespec="seconds"), "oleh": f"{NAMA} / {NIM}",
              "sumber": ["berkas: " + str(DATASET_FILE) + " (" + str(DATASET_URL) + ")", "API: Open-Meteo", ("scraping: " + SCRAPE_PAKAI) if SCRAPE_PAKAI else "scraping: tidak dipakai (tidak ada tabel statis yang lolos)"],
              "kunci_merge": "kunci_wilayah (nama kabupaten/kota ternormalisasi)", "tipe_merge": "inner, validate=one_to_one",
              "jumlah_baris": int(len(gabungan)), "kolom": list(gabungan.columns),
              "md5_gabungan_csv": md5_file(PROC / "gabungan_pariwisata_bali.csv")}
(DOCS / "metadata_gabungan.json").write_text(json.dumps(meta_merge, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(meta_merge, indent=2, ensure_ascii=False))

In [ ]:
# Visualisasi hasil gabungan
print("Wilayah dalam gabungan:", len(gabungan))
g = gabungan.sort_values("indikator_pariwisata", ascending=False)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
ax[0].bar(g["kunci_wilayah"].str.title(), g["indikator_pariwisata"], color="#2a6f97")
ax[0].set_title(f"Indikator pariwisata ({KOL_NILAI}) per wilayah"); ax[0].tick_params(axis="x", rotation=45)
sc = ax[1].scatter(g["hujan_total_mm"], g["indikator_pariwisata"], s=(g["scrape_1"]/g["scrape_1"].max()*600+40) if "scrape_1" in g else 120, c=g["suhu_rata2_C"], cmap="viridis")
for _, rr in g.iterrows(): ax[1].annotate(rr["kunci_wilayah"].title(), (rr["hujan_total_mm"], rr["indikator_pariwisata"]), fontsize=8, xytext=(4, 4), textcoords="offset points")
ax[1].set_xlabel("Curah hujan total 2025 (mm)"); ax[1].set_ylabel("Indikator pariwisata"); ax[1].set_title("Hujan vs pariwisata (ukuran=scrape_1, warna=suhu)")
plt.colorbar(sc, ax=ax[1], label="Suhu rata-rata (°C)")
plt.tight_layout(); plt.savefig(PROC / "visual_gabungan.png", dpi=150); plt.show()
print("Korelasi (Pearson):"); display(gabungan[["indikator_pariwisata"] + list(KET_KOLOM_SCRAPE) + ["suhu_rata2_C", "hujan_total_mm"]].corr().round(2))

### Kesimpulan & catatan
- Sumber (berkas, API, dan scraping bila berhasil) digabung lewat kunci wilayah dengan `validate="one_to_one"`; tidak ada duplikasi baris.
- Keterbatasan: n = 9 wilayah (korelasi tidak untuk kesimpulan kausal); cuaca dari titik ibu kota; tahun dataset berkas bisa berbeda dengan 2025.
- Atribusi: Open-Meteo (CC BY 4.0), situs pemerintah sumber scraping (lihat `docs/scraping_meta.json`), dataset portal sesuai lisensinya.

## Unduh hasil (untuk upload ke GitHub / ELSA)
Mengemas `data/raw`, `Process`, dan `docs` menjadi satu zip. Di Colab, zip langsung terunduh.

In [ ]:
import shutil
shutil.make_archive("hasil_UTS_DAM_2501010174", "zip", root_dir=".", base_dir="data")
shutil.make_archive("docs_UTS_DAM_2501010174", "zip", root_dir=".", base_dir="docs")
if IN_COLAB:
    from google.colab import files
    files.download("hasil_UTS_DAM_2501010174.zip"); files.download("docs_UTS_DAM_2501010174.zip")